# Battery deployment 2026 vs 2025

In [6]:
import pandas as pd
import pathlib
import mlflow
import sklearn.neighbors
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [7]:
def read_data(rel_path):
    path = pathlib.Path("../data").resolve() / (rel_path + ".parquet")
    df = pd.read_parquet(path)
    return df

In [8]:
data_sessions_2026 = read_data("gold/session_metadata_Y2026")
data_sessions_2025 = read_data("gold/session_metadata_Y2025")


In [9]:
data_sessions_2026[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Miami Gardens
20,0,Bahrain


In [10]:
data_sessions_2025[['round_number', 'event_location']].drop_duplicates()

,round_number,event_location
0,1,Melbourne
5,2,Shanghai
10,3,Suzuka
15,4,Sakhir
20,5,Jeddah
25,6,Miami Gardens
30,7,Imola
35,8,Monaco
40,9,Barcelona
45,10,Montréal


In [11]:
data_laps_2026 = read_data("gold/session_laps_Y2026")
data_laps_2025 = read_data("gold/session_laps_Y2025")


In [12]:
data_telemetry_pos_2026 = read_data("gold/telemetry_pos_Y2026R03")
data_telemetry_pos_2025 = read_data("gold/telemetry_pos_Y2025R03")
data_telemetry_car_2026 = read_data("gold/telemetry_car_Y2026R03")
data_telemetry_car_2025 = read_data("gold/telemetry_car_Y2025R03")


In [13]:
data_telemetry_pos_2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status
0,2025,Y2025R03S1,1,1,2025-04-04 02:30:43.362000+00:00,940.878,0.154,1,3769.0,-3284.0,0.0,OnTrack
1,2025,Y2025R03S1,1,1,2025-04-04 02:30:43.562000+00:00,941.078,0.354,1,3794.0,-3311.0,0.0,OnTrack
2,2025,Y2025R03S1,1,1,2025-04-04 02:30:43.842000+00:00,941.358,0.634,1,3807.0,-3326.0,0.0,OnTrack
3,2025,Y2025R03S1,1,1,2025-04-04 02:30:44.042000+00:00,941.558,0.834,1,3825.0,-3346.0,0.0,OnTrack
4,2025,Y2025R03S1,1,1,2025-04-04 02:30:44.242000+00:00,941.758,1.034,1,3845.0,-3367.0,0.0,OnTrack
...,...,...,...,...,...,...,...,...,...,...,...,...
1336956,2025,Y2025R03S5,87,53,2025-04-06 06:26:47.317000+00:00,8346.887,91.017,1,853.0,286.0,777.0,OnTrack
1336957,2025,Y2025R03S5,87,53,2025-04-06 06:26:47.477000+00:00,8347.047,91.177,1,935.0,209.0,774.0,OnTrack
1336958,2025,Y2025R03S5,87,53,2025-04-06 06:26:47.677000+00:00,8347.247,91.377,1,1037.0,108.0,769.0,OnTrack
1336959,2025,Y2025R03S5,87,53,2025-04-06 06:26:47.877000+00:00,8347.447,91.577,1,1129.0,14.0,766.0,OnTrack


In [14]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2026R03S1_Y2026R03S2_Y2026R03S3_Y2026R03S4_Y2026R03S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2026  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': 'baf068f5fcc4410ab8f9e3ddebb2211b', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/baf068f5fcc4410ab8f9e3ddebb2211b/artifacts', 'start_time': Timestamp('2026-05-10 15:11:11.453000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-10 15:12:27.484000+0000', tz='UTC'), 'metrics.mae-time-z': 2.29225902202003, 'metrics.rmse-distance-z': 0.16518651136829954, 'metrics.mae-distance-x': 0.41026919072288554, 'metrics.rmse-time-z': 4.482068744659485, 'metrics.mae-distance-y': 0.45017308275740847, 'metrics.rmse-time-y': 71.47788760444536, 'metrics.adjustment': 9.542502054502799e-05, 'metrics.mae-time-y': 48.381918357545445, 'metrics.rmse-distance-x': 0.6363235904418081, 'metrics.mae-time-x': 57.035792652296045, 'metrics.mae-distance-z': 0.11612836752658146, 'metrics.rmse-distance-y': 0.7111431045035588, 'metrics.rmse-time-x': 82.91495288466035, 'params.max_degree': '100', 'params.predict_size': '100

In [15]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["formula_one_circuit_map"])
runs = runs[runs['tags.session_ids'].str.contains('Y2025R03S1_Y2025R03S2_Y2025R03S3_Y2025R03S4_Y2025R03S5')]
print(runs.iloc[0].to_dict())
df_circuit_map_2025  =  pd.read_parquet(
    mlflow.artifacts.download_artifacts(artifact_uri=runs.iloc[0]["artifact_uri"] + "/result.parquet")
)

{'run_id': '5c02b728fa48481c98a6dafaa22fdd58', 'experiment_id': '1', 'status': 'FINISHED', 'artifact_uri': '/Users/tiagobbatalhao/Documents/projects/formula_one_data_analysis/mlruns/1/5c02b728fa48481c98a6dafaa22fdd58/artifacts', 'start_time': Timestamp('2026-05-10 15:14:10.039000+0000', tz='UTC'), 'end_time': Timestamp('2026-05-10 15:15:25.104000+0000', tz='UTC'), 'metrics.mae-time-z': 2.0703993121682225, 'metrics.rmse-distance-z': 0.15951165954442834, 'metrics.mae-distance-x': 0.4059662139004732, 'metrics.rmse-time-z': 4.093554267444461, 'metrics.mae-distance-y': 0.420602595223019, 'metrics.rmse-time-y': 61.413565680307705, 'metrics.adjustment': -8.705981175124327e-05, 'metrics.mae-time-y': 42.577221351618306, 'metrics.rmse-distance-x': 0.6538965262155709, 'metrics.mae-time-x': 49.14156102295295, 'metrics.mae-distance-z': 0.1078392339060892, 'metrics.rmse-distance-y': 0.6647682616796105, 'metrics.rmse-time-x': 68.32064770375922, 'params.max_degree': '100', 'params.predict_size': '1000

In [16]:
def run_circuit_encoding(telemetry_pos: pd.DataFrame, circuit_map: pd.DataFrame) -> pd.DataFrame:
    columns_coordinates = ["coordinate_x", "coordinate_y"]
    columns_pkey = ["session_id", "driver_number", "lap_number", "timestamp"]
    find_neighbours = (
        sklearn.neighbors.NearestNeighbors(n_neighbors=1)
        .fit(circuit_map[columns_coordinates].values)
        .kneighbors(telemetry_pos[columns_coordinates].values)
    )
    df_pos = telemetry_pos[columns_pkey].copy()
    df_pos["idx"] = find_neighbours[1][:, 0]
    df_pos = df_pos.merge(
        circuit_map.assign(idx=lambda df: range(len(df)))[["idx", "encoding", "distance_m"]]
    )
    return df_pos


In [17]:
if "distance_m" not in data_telemetry_pos_2026.columns:
    data_telemetry_pos_2026 = data_telemetry_pos_2026.merge(
        run_circuit_encoding(data_telemetry_pos_2026, df_circuit_map_2026)
    )
if "distance_m" not in data_telemetry_pos_2025.columns:
    data_telemetry_pos_2025 = data_telemetry_pos_2025.merge(
        run_circuit_encoding(data_telemetry_pos_2025, df_circuit_map_2025)
    )


In [36]:
lap_2026 = data_laps_2026[
    (data_laps_2026['session_id']=='Y2026R03S4')
].sort_values(by=['time_lap'], ascending=True).iloc[1:2]
lap_2026.T

,6829
year,2026
session_id,Y2026R03S4
driver_number,12
driver_name,ANT
driver_team,Mercedes
lap_number,14
stint,5.0
timestamp_lap_start,NaT
timing_start_lap,4170.558
timing_end_lap,4259.397


In [37]:
lap_2025 = data_laps_2025[
    (data_laps_2025['session_id']=='Y2025R03S4')
].sort_values(by=['time_lap'], ascending=True).iloc[:1]
lap_2025.T

,6046
year,2025
session_id,Y2025R03S4
driver_number,1
driver_name,VER
driver_team,Red Bull Racing
lap_number,16
stint,6.0
timestamp_lap_start,2025-04-05 07:07:05.373000+00:00
timing_start_lap,4843.493
timing_end_lap,4930.476


In [38]:
data_telemetry_pos_lap2026 = (
    data_telemetry_pos_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2026 = (
    data_telemetry_car_2026.merge(
        lap_2026[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [39]:
data_telemetry_pos_lap2025 = (
    data_telemetry_pos_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)
data_telemetry_car_lap2025 = (
    data_telemetry_car_2025.merge(
        lap_2025[['session_id', 'driver_number', 'lap_number']], how='inner'
    )
)

In [40]:
data_telemetry_pos_lap2026

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2026,Y2026R03S4,12,14,2026-03-28 06:57:58.199000+00:00,4170.761,0.203,1,1695.0,-643.0,740.0,OnTrack,197,0.00197,11.347944
1,2026,Y2026R03S4,12,14,2026-03-28 06:57:58.399000+00:00,4170.961,0.403,1,1796.0,-761.0,735.0,OnTrack,467,0.00467,26.899234
2,2026,Y2026R03S4,12,14,2026-03-28 06:57:58.699000+00:00,4171.261,0.703,1,2012.0,-1014.0,726.0,OnTrack,1044,0.01044,60.148821
3,2026,Y2026R03S4,12,14,2026-03-28 06:57:59.079000+00:00,4171.641,1.083,1,2225.0,-1263.0,717.0,OnTrack,1613,0.01613,92.924413
4,2026,Y2026R03S4,12,14,2026-03-28 06:57:59.519000+00:00,4172.081,1.523,1,2387.0,-1453.0,710.0,OnTrack,2046,0.02046,117.877481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,2026,Y2026R03S4,12,14,2026-03-28 06:59:25.379000+00:00,4257.941,87.383,1,876.0,265.0,776.0,OnTrack,98073,0.98073,5650.052516
330,2026,Y2026R03S4,12,14,2026-03-28 06:59:25.599000+00:00,4258.161,87.603,1,990.0,156.0,771.0,OnTrack,98347,0.98347,5665.820463
331,2026,Y2026R03S4,12,14,2026-03-28 06:59:25.859000+00:00,4258.421,87.863,1,1155.0,-12.0,764.0,OnTrack,98756,0.98756,5689.383926
332,2026,Y2026R03S4,12,14,2026-03-28 06:59:26.059000+00:00,4258.621,88.063,1,1287.0,-164.0,758.0,OnTrack,99105,0.99105,5709.497220


In [41]:
data_telemetry_pos_lap2025

,year,session_id,driver_number,lap_number,timestamp,timing_from_session,timing_from_lap,track_status,coordinate_x,coordinate_y,coordinate_z,position_status,idx,encoding,distance_m
0,2025,Y2025R03S4,1,16,2025-04-05 07:07:05.475000+00:00,4843.595,0.102,1,1660.0,-602.0,741.0,OnTrack,50,0.00050,2.881512
1,2025,Y2025R03S4,1,16,2025-04-05 07:07:05.676000+00:00,4843.796,0.303,1,1763.0,-722.0,737.0,OnTrack,324,0.00324,18.684314
2,2025,Y2025R03S4,1,16,2025-04-05 07:07:06.055000+00:00,4844.175,0.682,1,1960.0,-953.0,728.0,OnTrack,851,0.00851,49.063436
3,2025,Y2025R03S4,1,16,2025-04-05 07:07:06.436000+00:00,4844.556,1.063,1,2182.0,-1213.0,719.0,OnTrack,1444,0.01444,83.256141
4,2025,Y2025R03S4,1,16,2025-04-05 07:07:06.716000+00:00,4844.836,1.343,1,2310.0,-1363.0,713.0,OnTrack,1786,0.01786,102.967144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,2025,Y2025R03S4,1,16,2025-04-05 07:08:31.336000+00:00,4929.456,85.963,1,1109.0,35.0,766.0,OnTrack,98589,0.98589,5683.237869
325,2025,Y2025R03S4,1,16,2025-04-05 07:08:31.515000+00:00,4929.635,86.142,1,1198.0,-60.0,762.0,OnTrack,98815,0.98815,5696.265112
326,2025,Y2025R03S4,1,16,2025-04-05 07:08:31.736000+00:00,4929.856,86.363,1,1298.0,-177.0,757.0,OnTrack,99082,0.99082,5711.646393
327,2025,Y2025R03S4,1,16,2025-04-05 07:08:31.956000+00:00,4930.076,86.583,1,1404.0,-303.0,752.0,OnTrack,99367,0.99367,5728.086443


In [42]:
_start_2026, _end_2026 = 3917, 5348
df1 = data_telemetry_pos_lap2026[
    (data_telemetry_pos_lap2026['distance_m'] > _start_2026 - 50)
    & (data_telemetry_pos_lap2026['distance_m'] < _end_2026 + 50)
]
df2 = data_telemetry_car_lap2026[
    (data_telemetry_car_lap2026['timing_from_lap'] > df1['timing_from_lap'].min())
    & (data_telemetry_car_lap2026['timing_from_lap'] < df1['timing_from_lap'].max())
]
df2['distance_m'] = np.interp(
    df2['timing_from_lap'],
    data_telemetry_pos_lap2026['timing_from_lap'],
    data_telemetry_pos_lap2026['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_80382/1791150302.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['distance_m'] = np.interp(


In [43]:
_start_2025, _end_2025 = 3917, 5348
df3 = data_telemetry_pos_lap2025[
    (data_telemetry_pos_lap2025['distance_m'] > _start_2025 - 50)
    & (data_telemetry_pos_lap2025['distance_m'] < _end_2025 + 50)
]
df4 = data_telemetry_car_lap2025[
    (data_telemetry_car_lap2025['timing_from_lap'] > df3['timing_from_lap'].min())
    & (data_telemetry_car_lap2025['timing_from_lap'] < df3['timing_from_lap'].max())
]
df4['distance_m'] = np.interp(
    df4['timing_from_lap'],
    data_telemetry_pos_lap2025['timing_from_lap'],
    data_telemetry_pos_lap2025['distance_m'],
)

/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_80382/3530592174.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['distance_m'] = np.interp(


In [45]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=["Speed", "Throttle"],
    # title="Telemetry data on the Shanghai straight",
)
plot_df = df2[(df2['distance_m']>_start_2026) & (df2['distance_m']<_end_2026 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2026',
        line=dict(color='blue'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2026',
        line=dict(color='blue'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=2, col=1,
)
plot_df = df4[(df4['distance_m']>_start_2025) & (df4['distance_m']<_end_2025 + 100)]
plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['speed'].values,
        name='Speed - 2025',
        line=dict(color='green'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df['distance_plt'].values,
        y=plot_df['throttle'].values,
        name='Throttle - 2025',
        line=dict(color='green'),
        marker=dict(size=4),
        mode='lines+markers',
    ),
    row=2, col=1,
)
fig.update_layout(
    title="Telemetry data on Suzuka (T14 to T16)",
    height=600,
    width=1000,
    hovermode="x unified",
    showlegend=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikedash="solid",
    spikecolor="rgba(128, 128, 128, 0.5)",
    spikethickness=1,
    # title="Distance to apex (m)"
)



/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_80382/3239834225.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2026
/var/folders/n9/qtf08bkx00z3xzx24pfblmhc0000gp/T/ipykernel_80382/3239834225.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_df['distance_plt'] = plot_df['distance_m'] - _end_2025
